In [1]:
import datetime
import random

class SuperSystemManager:
    def __init__(self):
        # Hardcoded database connection strings and state
        self.db_host = "127.0.0.1"
        self.db_user = "admin"
        self.db_pass = "password123"
        
        self.inventory = {"apple": 50, "laptop": 2, "book": 15, "headphones": 10}
        self.users = {"alice": "pass1", "bob": "pass2"}

    def do_everything(self, u, p, items, discount_code):
        # 1. Authentication Responsibility
        print(f"Connecting to db at {self.db_host} with {self.db_user}...")
        if u not in self.users or self.users[u] != p:
            print("Auth failed!")
            return False
        print("Auth success!")

        # 2. Inventory Check Responsibility
        total_price = 0.0
        items_to_buy = []
        for i in items:
            item_name = i[0]
            qty = i[1]
            if item_name in self.inventory and self.inventory[item_name] >= qty:
                print(f"Item {item_name} is in stock.")
                self.inventory[item_name] -= qty
                items_to_buy.append(i)
                
                # Hardcoded pricing logic ???
                if item_name == "apple":
                    total_price += 1.50 * qty
                elif item_name == "laptop":
                    total_price += 999.99 * qty
                elif item_name == "book":
                    total_price += 15.00 * qty
                elif item_name == "headphones":
                    total_price += 45.50 * qty
            else:
                print(f"Sorry, {item_name} is out of stock.")

        # 3. Discount & Tax Calculation Responsibility (Magic Numbers)
        if discount_code == "SAVE20":
            total_price = total_price * 0.80
            print("20% discount applied.")
        elif discount_code == "HALF":
            total_price = total_price * 0.50
            print("50% discount applied.")

        total_price = total_price * 1.08 # 8% Hardcoded Tax

        # 4. Payment Processing Responsibility
        print(f"Attempting to charge credit card for ${total_price:.2f}...")
        payment_success = random.choice([True, True, False]) # Simulating an external API
        if not payment_success:
            print("Payment declined! Reverting inventory...")
            for i in items_to_buy:
                self.inventory[i[0]] += i[1]
            return False
        
        print("Payment successful!")

        # 5. Database/File Logging Responsibility
        receipt_id = random.randint(1000, 9999)
        with open("transactions_log.txt", "a") as f:
            f.write(f"[{datetime.datetime.now()}] User: {u}, Receipt: {receipt_id}, Total: ${total_price:.2f}\n")
        print("Saved to transaction log.")

        # 6. Email Notification Responsibility
        print(f"Connecting to SMTP server to send email to {u}@example.com...")
        print(f"EMAIL BODY: Thank you {u}! Your order #{receipt_id} for ${total_price:.2f} is confirmed.")

        return True

# --- Main Execution ---
if __name__ == "__main__":
    system = SuperSystemManager()

    # User inputs are tightly coupled to the script execution
    print("Welcome to the store!")
    cart = [("laptop", 1), ("apple", 5), ("car", 1)] # 'car' doesn't exist

    # Executing the God Method
    success = system.do_everything("alice", "pass1", cart, "SAVE20")

    if success:
        print("Order completely finished!")
    else:
        print("Order failed.")


Welcome to the store!
Connecting to db at 127.0.0.1 with admin...
Auth success!
Item laptop is in stock.
Item apple is in stock.
Sorry, car is out of stock.
20% discount applied.
Attempting to charge credit card for $870.47...
Payment successful!
Saved to transaction log.
Connecting to SMTP server to send email to alice@example.com...
EMAIL BODY: Thank you alice! Your order #5169 for $870.47 is confirmed.
Order completely finished!


### Round 1

In [17]:
import datetime
import random

from dataclasses import dataclass
from enum import Enum, auto
from typing import Union, Optional


USERS = {"alice": "pass1", "bob": "pass2", "admin":"password123"}
TAX_AMOUNT = 0.08

@dataclass
class Credential:
    host:str
    user: str
    passwd: str

    def auth(self):
        print(f"Connecting to db at {self.host} with {self.user}...")
        if self.user not in USERS or USERS[self.user] != self.passwd:
            print("Auth failed!")
            return False
        print("Auth success!")
        return True


@dataclass
class InventoryItem:
    name:str
    amount: int
    unit_price:float


class Inventory:

    def __init__(self):
        self.state = [InventoryItem("apple",50,1.5),InventoryItem("laptop",2,999.99), InventoryItem("book",15, 15.), 
                     InventoryItem("headphones",10,45.50)] 

    def _find(self,name:str) -> InventoryItem|None:

        for s in self.state:
            if s.name == name:
                return s


        return None

    def have_inventory(self, name:str, qty:int) -> bool:
        if (i := self._find(name)) is not None:
            return i.amount >= qty
        else:
            return False

    def sell(self, name:str, qty:int):
        if (i := self._find(name)) is not None:
            i.amount -= qty

    def price(self,name) -> Optional[float]:
        if (i := self._find(name)) is not None:
            return i.unit_price
        else:
            return None
         

DISCOUNT_CODES = {"SAVE20": 0.8, "HALF":0.5}

def apply_discount(subtotal:float, code)->float:
    total_price = subtotal
    
    if (d := DISCOUNT_CODES.get(code)) is not None:
        total_price = total_price * d
        print(f"{(1-d)*100}% discount applied.")

    return total_price


class SuperSystemManager:
    def __init__(self, cred:Credential, inv:Inventory): 

        self.cred = cred
        self.inv = inv
        

    def do_everything(self, items:tuple[str,int], discount_code):

        if not cred.auth():
            return 

        total_price = 0.0
        items_to_buy = []
        for item_name, qty in items:
            if self.inv.have_inventory(item_name, qty):
                print(f"Item {item_name} is in stock.")

                self.inv.sell(item_name, qty) 
                total_price += self.inv.price(item_name) * qty 
                
                items_to_buy.append((item_name, qty))
                
            else:
                print(f"Sorry, {item_name} is out of stock.")


        total_price = apply_discount(total_price, discount_code)
        total_price = total_price * (1 + TAX_AMOUNT) 

        # 4. Payment Processing Responsibility
        print(f"Attempting to charge credit card for ${total_price:.2f}...")
        payment_success = random.choice([True, True, False])  # Simulating an external API
        if not payment_success:
            print("Payment declined! Reverting inventory...")
            for item_name, qty in items_to_buy:
                self.inv.sell(item_name, -qty)
            return False
        
        print("Payment successful!")

        # 5. Database/File Logging Responsibility
        receipt_id = random.randint(1000, 9999)
        with open("transactions_log.txt", "a") as f:
            f.write(f"[{datetime.datetime.now()}] User: {u}, Receipt: {receipt_id}, Total: ${total_price:.2f}\n")
        print("Saved to transaction log.")

        # 6. Email Notification Responsibility
        print(f"Connecting to SMTP server to send email to {u}@example.com...")
        print(f"EMAIL BODY: Thank you {u}! Your order #{receipt_id} for ${total_price:.2f} is confirmed.")

        return True

# --- Main Execution ---
if __name__ == "__main__":
    cred  = Credential(host="127.0.0.1",user="admin",passwd="password123")
    inv = Inventory()
    
    system = SuperSystemManager(cred, inv)

    # User inputs are tightly coupled to the script execution
    print("Welcome to the store!")
    cart = [("laptop", 1), ("apple", 5), ("car", 1)] # 'car' doesn't exist

    # Executing the God Method
    success = system.do_everything(cart, "SAVE20")

    if success:
        print("Order completely finished!")
    else:
        print("Order failed.")


Welcome to the store!
Connecting to db at 127.0.0.1 with admin...
Auth success!
Item laptop is in stock.
Item apple is in stock.
Sorry, car is out of stock.
19.999999999999996% discount applied.
Attempting to charge credit card for $870.47...
Payment declined! Reverting inventory...
Order failed.


### Round 2

In [25]:
import datetime
import random

from dataclasses import dataclass
from enum import Enum, auto

from typing import Protocol


USERS = {"alice": "pass1", "bob": "pass2", "admin":"password123"}
TAX_AMOUNT = 0.08

@dataclass
class Credential:
    host:str
    user: str
    passwd: str

    def auth(self):
        print(f"Connecting to db at {self.host} with {self.user}...")
        if self.user not in USERS or USERS[self.user] != self.passwd:
            print("Auth failed!")
            return False
        print("Auth success!")
        return True


@dataclass
class InventoryItem:
    name:str
    amount: int
    unit_price:float


class Notification(Protocol):
    def notify(self, receipt_id:int,user:str, amount:float):
        ...


class FileLogger:
    def __init__(self):
        self.file_name = "transactions_log.txt"
        
    def notify(self, receipt_id:int,user:str, amount:float):
        with open(self.file_name, "a") as f:
            f.write(f"[{datetime.datetime.now()}] User: {user}, Receipt: {receipt_id}, Total: ${amount:.2f}\n")
        print("Saved to transaction log.")

class EmailNotifier:
    def notify(self, receipt_id:int,user:str, amount:float):
        print(f"Connecting to SMTP server to send email to {user}@example.com...")
        print(f"EMAIL BODY: Thank you {user}! Your order #{receipt_id} for ${amount:.2f} is confirmed.")


class Inventory:

    def __init__(self):
        self.state = [InventoryItem("apple",50,1.5),InventoryItem("laptop",2,999.99), InventoryItem("book",15, 15.), 
                     InventoryItem("headphones",10,45.50)] 

    def _find(self,name:str) -> Optional[InventoryItem]:

        for s in self.state:
            if s.name == name:
                return s


        return None

    def have_inventory(self, name:str, qty:int) -> bool:
        if (i := self._find(name)) is not None:
            return i.amount >= qty
        else:
            return False

    def redo(self, name:str, qty:int):
        if (i := self._find(name)) is not None:
            i.amount -= qty
            
    def undo(self, name:str, qty:int):
        if (i := self._find(name)) is not None:
            i.amount += qty            

    def price(self,name) -> Optional[float]:
        if (i := self._find(name)) is not None:
            return i.unit_price
        else:
            return None
         

DISCOUNT_CODES = {"SAVE20": 0.8, "HALF":0.5}

def apply_discount(subtotal:float, discount_code)->float:
    total_price = subtotal
    
    if (d := DISCOUNT_CODES.get(discount_code)) is not None:
        total_price = total_price * d
        print(f"{(1-d)*100}% discount applied.")

    return total_price


class SuperSystemManager:
    def __init__(self, cred:Credential, inv:Inventory, services: list[Notification]): 

        self.cred = cred
        self.inv = inv
        self.services = services
        

    def do_everything(self, items:tuple[str,int], discount_code) -> bool:

        if not cred.auth():
            return False

        total_price = 0.0
        items_to_buy = []
        for item_name, qty in items:
            if self.inv.have_inventory(item_name, qty):
                print(f"Item {item_name} is in stock.")

                self.inv.redo(item_name, qty) 
                total_price += self.inv.price(item_name) * qty 
                
                items_to_buy.append((item_name, qty))
                
            else:
                print(f"Sorry, {item_name} is out of stock.")


        total_price = apply_discount(total_price, discount_code)
        total_price = total_price * (1 + TAX_AMOUNT) 

        # 4. Payment Processing Responsibility
        print(f"Attempting to charge credit card for ${total_price:.2f}...")
        payment_success = random.choice([True, True, False]) # Simulating an external API
        if not payment_success:
            print("Payment declined! Reverting inventory...")
            for item_name, qty in items_to_buy:
                self.inv.undo(item_name, qty) 
            return False
        
        print("Payment successful!")

        receipt_id = random.randint(1000, 9999)

        for srv in self.services:
            srv.notify(receipt_id, self.cred.user, total_price)
            
        return True

# --- Main Execution ---
if __name__ == "__main__":
    cred  = Credential(host="127.0.0.1",user="admin",passwd="password123")
    inv = Inventory()
    services = [EmailNotifier(), FileLogger()]
    
    system = SuperSystemManager(cred, inv,services)

    # User inputs are tightly coupled to the script execution
    print("Welcome to the store!")
    cart = [("laptop", 1), ("apple", 5), ("car", 1)] # 'car' doesn't exist

    # Executing the God Method
    success = system.do_everything( cart, "SAVE20")

    if success:
        print("Order completely finished!")
    else:
        print("Order failed.")


Welcome to the store!
Connecting to db at 127.0.0.1 with admin...
Auth success!
Item laptop is in stock.
Item apple is in stock.
Sorry, car is out of stock.
19.999999999999996% discount applied.
Attempting to charge credit card for $870.47...
Payment successful!
Connecting to SMTP server to send email to admin@example.com...
EMAIL BODY: Thank you admin! Your order #6940 for $870.47 is confirmed.
Saved to transaction log.
Order completely finished!
